# Data pre-processing
This study utilizes a subset of the Real-world Affective Faces Database (RAF-DB), a large-scale collection containing 15,000 highly diverse images across seven basic emotions: surprise, fear, disgust, happiness, sadness, anger, and neutral.

**Dataset Characteristics**
- **Origin:** Images were sourced from the Internet and labeled by approximately 40 independent annotators, ensuring high reliability and a realistic representation of age, gender, ethnicity, and lighting conditions.
- **Preprocessing:** The original images (located in ```\DATA\IMAGES\WITHOUT MASK```) consist of **ONLY** high-quality samples where facial landmarks and vectors were successfully extracted using Google Face Landmarker. These were normalized to $100 \times 100$ pixels and cropped.
- **Structure:** Data is split into training and testing sets (5:1 ratio). While standardized and compact enough for local hardware processing, the dataset is significantly imbalanced, with "Happiness" being the most frequent and "Fear" the least frequent class.

**Synthetic Data Generation**

To evaluate emotion recognition under occlusions, a synthetic dataset was created by applying surgical mask overlays to the normalized images. The processed output is stored in ```DATA\IMAGES\WITH MASK```. This approach allows for a controlled comparative analysis between masked and unmasked faces while maintaining original facial proportions and environmental consistency.

*After rejecting images of too low quality, a dataset was obtained that was reduced by approximately 900 images compared to the original RAF-DB set.*

<center><img src="DATA\OTHER\Class Distribution Before and After Filtering  Low-Quality.png"></center>

## Import libraries

In [ ]:
import os
import re

import cv2
import mediapipe as mp
import pandas as pd
from PIL import Image

## Import RAF-DB images
Only good quality images that are compatible with the Google Face Landmarks model.
1. Go to https://drive.google.com/drive/folders/1WrfXpQpv__KsxYh6objvyGOWs8Aj6fjq?usp=drive_link and download folder.
2. Move folder with all files to ```DATA\IMAGES```.

## IMAGE WITH MASK
Due to the lack of large, balanced datasets of masked faces, a synthetic dataset was generated using the **RAF-DB** collection. A custom Python script was developed to automatically overlay a consistent surgical mask graphic onto each image.

Leveraging the original dataset's standardized $100 \times 100$ pixel normalization and centered cropping, the script precisely covers the nose and mouth area while preserving critical upper-face features like eyes and eyebrows. This uniform approach ensures a controlled comparative analysis between masked and unmasked faces without environmental interference.

<center><img src="DATA\OTHER\mask application diagram.png"></center>

In [ ]:
FOLDERS = ['1', '2', '3', '4', '5', '6', '7'] # folder name == target
SPLITS = ['train', 'test']
MASK_IMAGE = Image.open("DATA\OTHER\mask image.png").convert("RGBA")

for split in SPLITS:
    for folder in FOLDERS:
        input_folder = os.path.join('DATA', 'IMAGES', 'WITHOUT MASK', split, folder)
        output_folder = os.path.join('DATA', 'IMAGES', 'WITH MASK', split, folder)
        os.makedirs(output_folder, exist_ok=True)
        for filename in os.listdir(input_folder):
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                input_path = os.path.join(input_folder, filename)
                image_cv = cv2.imread(input_path)
                if image_cv is None:
                    print(f"[!!!] Cannot load: {input_path}")
                    continue
                image_rgb = cv2.cvtColor(image_cv, cv2.COLOR_BGR2RGB)
                pil_image = Image.fromarray(image_rgb)
                img_width, img_height = pil_image.size
                mask_top = int(img_height / 2)
                mask_height = img_height - mask_top
                mask_resized = MASK_IMAGE.resize((img_width, mask_height))
                pil_image.paste(mask_resized, (0, mask_top), mask_resized)
                output_path = os.path.join(output_folder, filename)
                pil_image.save(output_path)

print("Mask application completed.")

Mask application completed.


# VECTOR WITHOUT MASK
Images were converted into vector face meshes using Google’s **MediaPipe Face Landmarker** model. This machine learning solution identifies **468 3D landmarks** ($x, y, z$ coordinates) and **1,322 connecting vectors**, enabling precise facial expression analysis and virtual avatar creation.

**Data Processing Pipeline**

The workflow follows a structured sequence: **Image → MP Face Landmarker → CSV/Excel file** containing landmark and vector data.

**Optimal Configuration**

The parameters were fine-tuned using the MediaPipe online tool to maximize the detection rate of the RAF-DB images while avoiding distortions from low-quality samples.

The final configuration used in the Python script is:

- ```static_image_mode=True```: 

Treats each image independently. While more computationally expensive, it ensures detection on every frame without the need for video tracking

- ```max_num_faces=1```: 

Limits processing to a single face per image, ideal for portrait datasets to increase speed and reduce errors.

- ```min_detection_confidence=0.34```: 

A threshold set to 34% to balance the inclusion of lower-quality images with the risk of misidentification.

- ```min_tracking_confidence=0.5```: 

Required for configuration, though effectively bypassed in static mode.

- ```refine_landmarks=False```: 

Uses the standard mesh output. Refining (e.g., for iris detection) was disabled to maintain consistency across lower-resolution images and prevent vector distortions.

<center><img src="DATA\OTHER\image-to-numeric data conversion scheme.png"></center>

In [3]:
def process_dataset(root_path: str, full_output_name: str, vector_output_name: str):
    """
    Processes images to extract face landmarks and edge distances.
    Saves two CSV files:
    1. Full data (Landmarks + Edges).
    2. Vectors only (Edges, without specific landmark coordinates).
    """
    mp_face_mesh = mp.solutions.face_mesh
    face_mesh = mp_face_mesh.FaceMesh(
        static_image_mode=True,
        max_num_faces=1,
        min_detection_confidence=0.34,
        min_tracking_confidence=0.5,
        refine_landmarks=False,
    )
    CONNECTIONS = mp_face_mesh.FACEMESH_TESSELATION
    data_rows = []
    stats = {}
    print(f"Processing: {root_path}")
    for root, dirs, files in os.walk(root_path):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                file_path = os.path.join(root, file).replace('\\', '/')
                relative_path = os.path.relpath(file_path, root_path)
                target = os.path.dirname(relative_path).replace('\\', '/')
                img = cv2.imread(file_path)
                if img is None:
                    print(f"[!!!] Cannot load: {file_path}")
                    continue
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                results = face_mesh.process(img_rgb)
                if target not in stats:
                    stats[target] = {'total': 0, 'detected': 0}
                stats[target]['total'] += 1
                row = {'filename': file, 'target': target}
                if results.multi_face_landmarks:
                    stats[target]['detected'] += 1
                    face_landmarks = results.multi_face_landmarks[0]
                    # 1. landmarks
                    for i, lm in enumerate(face_landmarks.landmark):
                        row[f'landmark_{i}_x'] = lm.x
                        row[f'landmark_{i}_y'] = lm.y
                    # 2. edges
                    for (start_idx, end_idx) in CONNECTIONS:
                        pt1 = face_landmarks.landmark[start_idx]
                        pt2 = face_landmarks.landmark[end_idx]
                        dx = pt1.x - pt2.x
                        dy = pt1.y - pt2.y
                        dz = pt1.z - pt2.z
                        dist = (dx**2 + dy**2 + dz**2) ** 0.5
                        idx1, idx2 = sorted((start_idx, end_idx))
                        label = f'edge_{idx1}_{idx2}'
                        row[label] = dist
                    data_rows.append(row)
                else:
                    print(f'[!!!] no face detected in file: {file_path}')


    # === SAVE DATA ===
    if not data_rows:
        print("[!!!] No data found to save.")
        return
    df = pd.DataFrame(data_rows)
    # Save Full Data (Landmarks + Vectors)
    os.makedirs(os.path.dirname(full_output_name), exist_ok=True)
    df.to_csv(f'{full_output_name}.csv', index=False)
    print(f"Saved Full Data: {full_output_name}.csv")
    # Save Vectors Only (Exclude columns starting with 'landmark_')
    cols_to_keep = [c for c in df.columns if not c.startswith('landmark_')]
    df_vectors = df[cols_to_keep]
    os.makedirs(os.path.dirname(vector_output_name), exist_ok=True)
    df_vectors.to_csv(f'{vector_output_name}.csv', index=False)
    print(f"Saved Vectors Only: {vector_output_name}.csv")
    # Print Summary
    print("\n--- Statistics ---")
    for subfolder, counts in stats.items():
        print(f"- {subfolder}: {counts['detected']}/{counts['total']} faces detected")
    print("-" * 20 + "\n")


# === EXECUTION ===
# Process Test Set
process_dataset(
    root_path='DATA/IMAGES/WITHOUT MASK/test',
    full_output_name='DATA/VECTORS AND LANDMARKS/without_mask_test',
    vector_output_name='DATA/VECTORS/without_mask_test'
)
# Process Train Set
process_dataset(
    root_path='DATA/IMAGES/WITHOUT MASK/train',
    full_output_name='DATA/VECTORS AND LANDMARKS/without_mask_train',
    vector_output_name='DATA/VECTORS/without_mask_train'
)

Processing: DATA/IMAGES/WITHOUT MASK/test
Saved Full Data: DATA/VECTORS AND LANDMARKS/without_mask_test.csv
Saved Vectors Only: DATA/VECTORS/without_mask_test.csv

--- Statistics ---
- 1: 315/315 faces detected
- 2: 66/66 faces detected
- 3: 159/159 faces detected
- 4: 1137/1137 faces detected
- 5: 442/442 faces detected
- 6: 144/144 faces detected
- 7: 612/612 faces detected
--------------------

Processing: DATA/IMAGES/WITHOUT MASK/train
Saved Full Data: DATA/VECTORS AND LANDMARKS/without_mask_train.csv
Saved Vectors Only: DATA/VECTORS/without_mask_train.csv

--- Statistics ---
- 1: 1225/1225 faces detected
- 2: 256/256 faces detected
- 3: 687/687 faces detected
- 4: 4606/4606 faces detected
- 5: 1841/1841 faces detected
- 6: 611/611 faces detected
- 7: 2315/2315 faces detected
--------------------



# VECTOR WITH MASK
Unfortunately, the **MediaPipe Face Landmarker** model was only able to successfully detect landmarks on approximately 50% of the masked images. This significant data loss would have compromised model training, particularly for underrepresented emotion classes.

To address this, a Python script was used to manually zero out the coordinates and vectors located in the area covered by the surgical mask, using the high-quality meshes previously extracted from the unmasked images. This approach simulates the absence of input data in the occluded region while maintaining the full size of the dataset.

Alternatively, removing the affected columns was considered; however, this would have altered the data format, preventing a direct comparison between masked and unmasked models. By retaining all columns and setting occluded values to zero, the dataset remains structurally consistent while realistically reflecting the loss of facial features.

<center><img src="DATA\OTHER\diagram of applying a mask to numerical data.png" width=500 hight=500></center>

The vectors and points that the mask covers are in the file: ```DATA\OTHER\data covered by a face mask.csv```

In [4]:
EDGES_PATH = 'DATA/OTHER/data covered by a face mask.csv'
df_edges = pd.read_csv(EDGES_PATH)
# Collect unique points as strings
points = set(df_edges['point1'].astype(str)).union(df_edges['point2'].astype(str))


def is_target_column(column_name, target_points):
    """
    Check if the column name contains any of the target points 
    using a regular expression.
    """
    for point in target_points:
        if re.search(rf'_(?:{point})(?:_|$)', column_name):
            return True
    return False


def process_file(input_path, output_path):
    """
    Reads a CSV, sets columns matching target points to zero, 
    and saves as an Excel file.
    """
    if not os.path.exists(input_path):
        print(f"[!!!] Warning: File not found: {input_path}")
        return
    df = pd.read_csv(input_path)
    # Overwrite matching columns with zeros
    for col in df.columns:
        if is_target_column(col, points):
            df[col] = 0
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f"Processed: {output_path}")


# === EXECUTION ===
directories = ['DATA/VECTORS', 'DATA/VECTORS AND LANDMARKS']
modes = ['test', 'train']
for directory in directories:
    for mode in modes:
        input_file = f"{directory}/without_mask_{mode}.csv"
        output_file = f"{directory}/with_mask_{mode}.csv"
        process_file(input_file, output_file)

Processed: DATA/VECTORS/with_mask_test.csv
Processed: DATA/VECTORS/with_mask_train.csv
Processed: DATA/VECTORS AND LANDMARKS/with_mask_test.csv
Processed: DATA/VECTORS AND LANDMARKS/with_mask_train.csv
